# LeetCode 3303 — Find the Occurrence of First Almost Equal Substring bằng Python

- Link: https://leetcode.com/problems/find-the-occurrence-of-first-almost-equal-substring/
- Độ khó: **Hard**
- Chủ đề: **Z-function** (độ dài khớp tiền tố tại mọi vị trí) trên `pattern + '#' + s` và trên bản **đảo ngược** (khớp hậu tố); điều kiện "sai khác ≤ 1" ⇔ `khớp_tiền_tố + khớp_hậu_tố ≥ m − 1`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minStartingIndex(self, s: str, pattern: str) -> int:
```

## 1. Mục tiêu học

1. Chuỗi `x` "gần bằng" `y` (cùng độ dài) ⇔ khác nhau tại **tối đa một** vị trí.
2. Với đoạn `s[i..i+m−1]`: gọi `L` = độ dài khớp từ trái với `pattern`, `R` = độ dài khớp từ phải. Sai khác ≤ 1 ⇔ `L + R ≥ m − 1` (chỉ còn tối đa một vị trí ở giữa chưa khớp).
3. Tính `L` cho mọi `i` bằng **Z-function** trên `pattern + '#' + s`; tính `R` bằng Z-function trên chuỗi **đảo ngược**.
4. Cách thay thế: băm + tìm kiếm nhị phân LCP (mở rộng được cho "≤ k sai khác").
5. Kiểm chứng bằng vét cạn.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và `pattern`. Chuỗi `x` **gần bằng** `y` nếu đổi tối đa một ký tự của `x` thì thành `y`. Trả về **chỉ số bắt đầu nhỏ nhất** của một xâu con của `s` gần bằng `pattern`, hoặc `-1`.

**Ví dụ 1:** `s = "abcdefg"`, `pattern = "bcdffg"` → `1`. **Ví dụ 2:** `s = "ababbababa"`, `pattern = "bacaba"` → `4`.
**Ví dụ 3:** `s = "abcd"`, `pattern = "dba"` → `-1`. **Ví dụ 4:** `s = "dde"`, `pattern = "d"` → `0`.

**Ràng buộc:** `1 ≤ len(pattern) < len(s) ≤ 10^5`; chữ thường.

**Follow-up:** nếu được đổi tối đa `k` ký tự **liên tiếp** thì sao?

In [1]:
from typing import List

examples = [("abcdefg", "bcdffg", 1), ("ababbababa", "bacaba", 4), ("abcd", "dba", -1), ("dde", "d", 0)]
for s, p, exp in examples:
    print(repr(s), repr(p), '-> kỳ vọng', exp)

'abcdefg' 'bcdffg' -> kỳ vọng 1
'ababbababa' 'bacaba' -> kỳ vọng 4
'abcd' 'dba' -> kỳ vọng -1
'dde' 'd' -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Z-function

`Z[i]` = độ dài tiền tố chung dài nhất của `t` và `t[i:]`. Thuật toán `O(|t|)` duy trì "hộp" `[l, r)` là đoạn khớp tiền tố xa nhất đã biết.

In [2]:
def z_function(t):
    n = len(t)
    z = [0] * n
    if n:
        z[0] = n
    l = r = 0
    for i in range(1, n):
        if i < r:
            z[i] = min(r - i, z[i - l])
        while i + z[i] < n and t[z[i]] == t[i + z[i]]:
            z[i] += 1
        if i + z[i] > r:
            l, r = i, i + z[i]
    return z

print(z_function("aabxaab"))

[7, 1, 0, 0, 3, 1, 0]


### 3.2. Khớp tiền tố tại mọi vị trí của `s`

Với `t = pattern + '#' + s` (ký tự `#` không xuất hiện trong chữ thường), `pre[i] = Z[m + 1 + i]` = số ký tự đầu của `pattern` khớp với `s` bắt đầu tại `i` (không vượt `m` nhờ `#`).

### 3.3. Khớp hậu tố

Đảo ngược cả hai: `t' = rev(pattern) + '#' + rev(s)`. Với đoạn `s[i..i+m−1]`, vị trí kết thúc `e = i + m − 1` ứng với chỉ số `n − 1 − e` trong `rev(s)` ⇒ `suf[e] = Z'[m + 1 + (n − 1 − e)]` = số ký tự **cuối** của `pattern` khớp với `s` kết thúc tại `e`.

In [3]:
def match_arrays(s, p):
    n, m = len(s), len(p)
    z1 = z_function(p + '#' + s)
    z2 = z_function(p[::-1] + '#' + s[::-1])
    pre = [z1[m + 1 + i] for i in range(n)]
    suf = [z2[m + 1 + (n - 1 - e)] for e in range(n)]
    return pre, suf

pre, suf = match_arrays("abcdefg", "bcdffg")
print('pre =', pre)
print('suf =', suf)

pre = [0, 3, 0, 0, 0, 0, 0]
suf = [0, 0, 0, 0, 0, 0, 2]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `pattern = "bcdffg"` (`m = 6`), xét `i = 1`: `s[1..6] = "bcdefg"`.

- Khớp từ trái: `b c d` rồi `e ≠ f` ⇒ `L = pre[1] = 3`.
- Khớp từ phải: `g f` … `f`? `s[6] = g = g`, `s[5] = f = f`, `s[4] = e ≠ f` ⇒ `R = suf[6] = 2`.
- `L + R = 5 = m − 1` ⇒ đúng một vị trí sai (vị trí giữa) ⇒ gần bằng ✓.

Nếu `L + R ≥ m` thì khớp hoàn toàn (0 sai khác) — cũng hợp lệ.

## 5. Cách 1 — Brute Force: đếm sai khác từng vị trí

In [4]:
def brute(s, p):
    n, m = len(s), len(p)
    for i in range(n - m + 1):
        diff = 0
        for a, b in zip(s[i:i + m], p):
            if a != b:
                diff += 1
                if diff > 1:
                    break
        if diff <= 1:
            return i
    return -1

for s, p, exp in examples:
    print(brute(s, p), '| kỳ vọng', exp)

1 | kỳ vọng 1
4 | kỳ vọng 4
-1 | kỳ vọng -1
0 | kỳ vọng 0


**Độ phức tạp:** `O(n · m)` trường hợp xấu (ví dụ `s = "aaaa…a"`, `pattern = "aa…ab b"`).

## 6. Ý tưởng thuật toán / bước đột phá

```
pre, suf = match_arrays(s, pattern)
for i in 0..n−m:
    if pre[i] + suf[i + m − 1] >= m − 1: return i
return −1
```

Hai lần Z-function `O(n + m)`, một lần duyệt `O(n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Z-function hai chiều (lời giải chính)

In [5]:
def z_solution(s, p):
    n, m = len(s), len(p)
    pre, suf = match_arrays(s, p)
    for i in range(n - m + 1):
        if pre[i] + suf[i + m - 1] >= m - 1:
            return i
    return -1

for s, p, exp in examples:
    print(z_solution(s, p), '| kỳ vọng', exp)

1 | kỳ vọng 1
4 | kỳ vọng 4
-1 | kỳ vọng -1
0 | kỳ vọng 0


### 7.2. Cách 3 — Băm đa thức + tìm kiếm nhị phân LCP

Với mỗi `i`: tìm sai khác đầu tiên bằng nhị phân trên độ dài khớp (so hash), nhảy qua vị trí sai đó, rồi kiểm tra phần còn lại có khớp **hoàn toàn** không (một lần so hash). `O(n log m)`. Cách này mở rộng tự nhiên cho "tối đa `k` sai khác" (nhảy `k` lần).

In [6]:
def hash_solution(s, p, k=1):
    n, m = len(s), len(p)
    MOD, B = (1 << 61) - 1, 131
    pw = [1] * (n + 1)
    for i in range(n):
        pw[i + 1] = pw[i] * B % MOD
    def prefix_hash(t):
        h = [0] * (len(t) + 1)
        for i, c in enumerate(t):
            h[i + 1] = (h[i] * B + ord(c)) % MOD
        return h
    hs, hp = prefix_hash(s), prefix_hash(p)
    get = lambda h, l, r: (h[r] - h[l] * pw[r - l]) % MOD        # hash của [l, r)
    def lcp(i, j):                                              # khớp s[i:] với p[j:]
        lo, hi = 0, m - j
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if get(hs, i, i + mid) == get(hp, j, j + mid):
                lo = mid
            else:
                hi = mid - 1
        return lo
    for i in range(n - m + 1):
        j, used = 0, 0
        while True:
            j += lcp(i + j, j)
            if j >= m:
                return i
            used += 1
            if used > k:
                break
            j += 1                                             # bỏ qua vị trí sai
            if j >= m:
                return i
    return -1

for s, p, exp in examples:
    print(hash_solution(s, p), '| kỳ vọng', exp)

1 | kỳ vọng 1
4 | kỳ vọng 4
-1 | kỳ vọng -1
0 | kỳ vọng 0


### 7.3. Cách 4 — Follow-up: đổi tối đa `k` ký tự **liên tiếp**

Đoạn `s[i..i+m−1]` hợp lệ ⇔ các vị trí sai nằm trong một cửa sổ độ dài `k` ⇔ `pre[i] + suf[i + m − 1] ≥ m − k`. Cùng hai mảng Z, chỉ đổi ngưỡng!

In [7]:
def consecutive_k(s, p, k):
    n, m = len(s), len(p)
    pre, suf = match_arrays(s, p)
    for i in range(n - m + 1):
        if pre[i] + suf[i + m - 1] >= m - k:
            return i
    return -1

def consecutive_k_brute(s, p, k):
    n, m = len(s), len(p)
    for i in range(n - m + 1):
        bad = [j for j in range(m) if s[i + j] != p[j]]
        if not bad or bad[-1] - bad[0] + 1 <= k:
            return i
    return -1

print(consecutive_k("abcdefg", "bxyzfg", 3), consecutive_k_brute("abcdefg", "bxyzfg", 3))

1 1


## 8. TRACE từng bước

Trace cách 2 trên ví dụ 2: `s = "ababbababa"`, `pattern = "bacaba"`.

In [8]:
def trace(s, p):
    n, m = len(s), len(p)
    pre, suf = match_arrays(s, p)
    for i in range(n - m + 1):
        L, R = pre[i], suf[i + m - 1]
        ok = L + R >= m - 1
        print(f"i = {i}: đoạn {s[i:i + m]!r}, L = {L}, R = {R}, L + R = {L + R} {'>=' if ok else '<'} {m - 1}")
        if ok:
            print('đáp án:', i, '| vét cạn:', brute(s, p))
            return

trace("ababbababa", "bacaba")

i = 0: đoạn 'ababba', L = 0, R = 2, L + R = 2 < 5
i = 1: đoạn 'babbab', L = 2, R = 0, L + R = 2 < 5
i = 2: đoạn 'abbaba', L = 0, R = 3, L + R = 3 < 5
i = 3: đoạn 'bbabab', L = 1, R = 0, L + R = 1 < 5
i = 4: đoạn 'bababa', L = 2, R = 3, L + R = 5 >= 5
đáp án: 4 | vét cạn: 4


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def minStartingIndex(self, s: str, pattern: str) -> int:
        def zf(t):
            n = len(t)
            z = [0] * n
            l = r = 0
            for i in range(1, n):
                if i < r:
                    z[i] = min(r - i, z[i - l])
                while i + z[i] < n and t[z[i]] == t[i + z[i]]:
                    z[i] += 1
                if i + z[i] > r:
                    l, r = i, i + z[i]
            return z

        n, m = len(s), len(pattern)
        z1 = zf(pattern + '#' + s)
        z2 = zf(pattern[::-1] + '#' + s[::-1])
        for i in range(n - m + 1):
            # z1[m+1+i]: khớp từ trái; z2[m+1+(n-1-(i+m-1))] = z2[2m+... ]: khớp từ phải
            if z1[m + 1 + i] + z2[n - i + 1] >= m - 1:
                return i
        return -1

for s, p, exp in examples:
    got = Solution().minStartingIndex(s, p)
    print(got, 'OK' if got == exp else 'SAI')

1 OK
4 OK
-1 OK
0 OK


Chỉ số hậu tố: `m + 1 + (n − 1 − (i + m − 1)) = n − i + 1`.

## 10. Vì sao thuật toán đúng

Đặt `x = s[i..i+m−1]`, `L` = tiền tố chung của `x` và `pattern` (dừng ở sai khác đầu tiên), `R` = hậu tố chung.

- Nếu số sai khác `≤ 1`: không sai ⇒ `L = m`; một sai tại `j` ⇒ `L = j`, `R = m − 1 − j` ⇒ `L + R = m − 1`.
- Ngược lại `L + R ≥ m − 1` ⇒ mọi vị trí `< L` và `≥ m − R` đều khớp; phần còn lại có độ dài `≤ 1` ⇒ tối đa một sai khác.

Z-function với dấu phân cách `#` cho đúng `L` (không vượt `m`), tương tự cho `R` trên chuỗi đảo. Duyệt `i` tăng dần ⇒ trả về chỉ số nhỏ nhất.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n m)` | `O(1)` |
| 2. Z hai chiều | `O(n + m)` | `O(n + m)` |
| 3. Băm + nhị phân | `O(n log m)` (với `k = 1`) | `O(n)` |
| 4. Follow-up liên tiếp | `O(n + m)` | `O(n + m)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'z_solution': z_solution, 'hash_solution': hash_solution,
         'submit': lambda s, p: Solution().minStartingIndex(s, p)}
edge = [("ab", "a", 0), ("ab", "c", 0), ("ba", "ab", -1), ("xyz", "ab", -1), ("aaab", "bb", 2), ("abc", "ba", 1)]
for s, p, exp in examples + edge:
    assert brute(s, p) == exp, (s, p, brute(s, p))
    for name, f in impls.items():
        assert f(s, p) == exp, (name, s, p)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Lưu ý `("ab", "c")` → `0`: pattern một ký tự luôn "gần bằng" mọi ký tự (đổi một ký tự).

### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3303)
cnt = 0
for _ in range(1500):
    n = rng.randint(2, 14)
    s = ''.join(rng.choice('ab') for _ in range(n))
    p = ''.join(rng.choice('ab') for _ in range(rng.randint(1, n - 1)))
    exp = brute(s, p)
    for name, f in impls.items():
        assert f(s, p) == exp, (name, s, p)
    k = rng.randint(0, 3)
    assert consecutive_k(s, p, k) == consecutive_k_brute(s, p, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test (kể cả follow-up).')

Khớp vét cạn trên 1500 bộ test (kể cả follow-up).


### 12.3. Ca lớn `n = 10^5` (ca xấu cho vét cạn)

In [12]:
import time
N = 10 ** 5
big = {
    'a…a vs a…abb (không có)': ('a' * N, 'a' * (N // 2) + 'bb'),
    'a…ab vs a…acb (cuối)': ('a' * (N - 1) + 'b', 'a' * (N // 2 - 2) + 'cb'),
    'ngẫu nhiên ab': (''.join(rng.choice('ab') for _ in range(N)), ''.join(rng.choice('ab') for _ in range(20))),
}
for name, (s, p) in big.items():
    t = time.time(); a = Solution().minStartingIndex(s, p); t1 = time.time() - t
    t = time.time(); b = z_solution(s, p); t2 = time.time() - t
    assert a == b
    print(f"{name:26s}: {a:7d} | nộp {t1:.2f}s | cách 2 {t2:.2f}s")
assert Solution().minStartingIndex('a' * N, 'a' * (N // 2) + 'bb') == -1
assert Solution().minStartingIndex('a' * (N - 1) + 'b', 'a' * (N // 2 - 2) + 'cb') == N - (N // 2)
print('Kiểm tra độc lập hai ca đầu: đúng.')

a…a vs a…abb (không có)   :      -1 | nộp 0.08s | cách 2 0.09s
a…ab vs a…acb (cuối)      :   50000 | nộp 0.08s | cách 2 0.12s


ngẫu nhiên ab             :   12557 | nộp 0.06s | cách 2 0.06s
Kiểm tra độc lập hai ca đầu: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ tìm khớp **chính xác**

In [13]:
def wrong_exact(s, p):
    return s.find(p)                                          # SAI: không cho phép 1 sai khác

print('sai :', wrong_exact("abcdefg", "bcdffg"), '| đúng:', Solution().minStartingIndex("abcdefg", "bcdffg"))

sai : -1 | đúng: 1


**Vì sao sai:** đề cho phép đổi **một** ký tự.

### Lỗi 2 — Chỉ dùng khớp tiền tố (`pre[i] ≥ m − 1`)

In [14]:
def wrong_prefix_only(s, p):
    n, m = len(s), len(p)
    pre, _ = match_arrays(s, p)
    for i in range(n - m + 1):
        if pre[i] >= m - 1:                                  # SAI: chỉ cho phép sai ở vị trí cuối
            return i
    return -1

print('sai :', wrong_prefix_only("abcdefg", "bcdffg"), '| đúng:', Solution().minStartingIndex("abcdefg", "bcdffg"))

sai : -1 | đúng: 1


**Vì sao sai:** vị trí sai có thể ở **giữa** (ví dụ 1 sai ở vị trí thứ 4 của pattern); cần cả khớp từ phải.

### Lỗi 3 — Lệch chỉ số khi lấy khớp hậu tố

In [15]:
def wrong_suffix_index(s, p):
    n, m = len(s), len(p)
    pre, suf = match_arrays(s, p)
    for i in range(n - m + 1):
        e = min(i + m, n - 1)                                # SAI: vị trí kết thúc là i + m - 1
        if pre[i] + suf[e] >= m - 1:
            return i
    return -1

print('sai :', wrong_suffix_index("aba", "bb"), '| đúng:', Solution().minStartingIndex("aba", "bb"))

sai : 1 | đúng: 0


**Vì sao sai:** dùng khớp hậu tố của đoạn kết thúc tại `i + m` (lệch một ô) nên đánh giá sai đoạn. Với `s = "aba"`, `pattern = "bb"`: đoạn `"ab"` (i = 0) khác đúng một vị trí nên đáp án là `0`, nhưng bản sai ghép khớp tiền tố của `i = 0` (bằng 0) với khớp hậu tố tại vị trí 2 (`'a'`, bằng 0) ⇒ bỏ lỡ và trả về `1`.

### Lỗi 4 — So sánh theo **tập ký tự / số lượng** thay vì theo vị trí

In [16]:
from collections import Counter
def wrong_anagram(s, p):
    n, m = len(s), len(p)
    for i in range(n - m + 1):
        d = Counter(s[i:i + m]); d.subtract(Counter(p))
        if sum(abs(v) for v in d.values()) <= 2:             # SAI: hoán vị cũng được tính là gần bằng
            return i
    return -1

print('sai :', wrong_anagram("abc", "ba"), '| đúng:', Solution().minStartingIndex("abc", "ba"))

sai : 0 | đúng: 1


**Vì sao sai:** `"ab"` và `"ba"` cùng số lượng ký tự nhưng khác nhau ở **hai** vị trí — không gần bằng. Đáp án là `1` (`"bc"` vs `"ba"`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Khớp xâu cho phép ≤ 1 sai khác" ⇒ khớp tiền tố + khớp hậu tố (Z-function / prefix function trên chuỗi đảo).
- Ghép `pattern + '#' + text` để Z-function trả về độ dài khớp với pattern tại mọi vị trí của text.
- "≤ k sai khác" tổng quát ⇒ nhảy LCP `k + 1` lần (băm + nhị phân, hoặc mảng hậu tố + LCP RMQ).

## 15. Biến thể của bài

1. Đếm **tất cả** vị trí gần bằng (không chỉ vị trí đầu).
2. Tối đa `k` sai khác (không cần liên tiếp) ⇒ cách 3 với `k` tổng quát, `O(n k log m)`.
3. Follow-up đề: `k` ký tự **liên tiếp** ⇒ cách 4.
4. Vị trí gần bằng **cuối cùng** ⇒ duyệt ngược.

In [17]:
# Biến thể 1 và 2
def all_positions(s, p):
    n, m = len(s), len(p)
    pre, suf = match_arrays(s, p)
    return [i for i in range(n - m + 1) if pre[i] + suf[i + m - 1] >= m - 1]

def k_mismatch_brute(s, p, k):
    n, m = len(s), len(p)
    for i in range(n - m + 1):
        if sum(a != b for a, b in zip(s[i:i + m], p)) <= k:
            return i
    return -1

rng = random.Random(15)
for _ in range(400):
    n = rng.randint(2, 12)
    s = ''.join(rng.choice('abc') for _ in range(n))
    p = ''.join(rng.choice('abc') for _ in range(rng.randint(1, n - 1)))
    assert all_positions(s, p) == [i for i in range(n - len(p) + 1)
                                   if sum(a != b for a, b in zip(s[i:i + len(p)], p)) <= 1]
    k = rng.randint(0, 3)
    assert hash_solution(s, p, k) == k_mismatch_brute(s, p, k)
print('Biến thể 1 (mọi vị trí) và 2 (≤ k sai khác bằng băm) khớp vét cạn.')

Biến thể 1 (mọi vị trí) và 2 (≤ k sai khác bằng băm) khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Đếm sai khác | `O(nm)` | Oracle |
| 2 | Z hai chiều | `O(n + m)` | **Nộp bài** |
| 3 | Băm + nhị phân LCP | `O(n k log m)` | Tổng quát `k` sai khác |
| 4 | Z + ngưỡng `m − k` | `O(n + m)` | Follow-up liên tiếp |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `s = "zzz"`, `pattern = "ab"`.

**Bài 2 (tính tay):** `s = "abab"`, `pattern = "bb"`.

**Bài 3 (điền TODO):** trả về luôn **vị trí sai khác** (hoặc `-1` nếu khớp hoàn toàn) cho chỉ số tìm được.

**Bài 4 (tự viết lại):** đếm số vị trí `pattern` xuất hiện **chính xác** trong `s` bằng **prefix function (KMP)**.

**Bài 5 (mini challenge):** chỉ số nhỏ nhất mà đoạn gần bằng `pattern` **hoặc** gần bằng `reverse(pattern)`.

In [18]:
def mismatch_position_todo(s, p):
    # TODO
    raise NotImplementedError

def count_exact_kmp_todo(s, p):
    # TODO
    raise NotImplementedError

def either_direction_todo(s, p):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `"zz"` vs `"ab"`: 2 sai. (2) Mọi đoạn đều vậy. (3) `-1`.

**Bài 2.** (1) `"ab"`: 1 sai. (2) Hợp lệ ngay `i = 0`. (3) `0`.

**Bài 3.** (1) `L = pre[i]`. (2) Nếu `L = m` thì khớp hoàn toàn. (3) Ngược lại vị trí sai là `i + L`.

**Bài 4.** (1) Tính prefix function của `p + '#' + s`. (2) Tại vị trí ứng với `s[e]`, giá trị bằng `m` ⇔ `pattern` kết thúc tại `e`. (3) Đếm; đối chiếu với Z-function (`pre[i] == m`).

**Bài 5.** (1) Chạy lời giải hai lần. (2) Lấy min các kết quả khác `-1`. (3) Vét cạn.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().minStartingIndex("zzz", "ab") == -1
assert Solution().minStartingIndex("abab", "bb") == 0

def mismatch_position(s, p):
    i = Solution().minStartingIndex(s, p)
    if i < 0:
        return i, None
    pre, _ = match_arrays(s, p)
    return i, (None if pre[i] >= len(p) else i + pre[i])

print('Bài 1: -1 | Bài 2: 0 | Bài 3:', mismatch_position("abcdefg", "bcdffg"), '(s[4] = e cần đổi)')

Bài 1: -1 | Bài 2: 0 | Bài 3: (1, 4) (s[4] = e cần đổi)


In [20]:
# Bài 4, 5
def prefix_function(t):
    pi = [0] * len(t)
    for i in range(1, len(t)):
        j = pi[i - 1]
        while j and t[i] != t[j]:
            j = pi[j - 1]
        if t[i] == t[j]:
            j += 1
        pi[i] = j
    return pi

def count_exact_kmp(s, p):
    m = len(p)
    pi = prefix_function(p + '#' + s)
    return sum(1 for e in range(len(s)) if pi[m + 1 + e] == m)

def either_direction(s, p):
    a, b = Solution().minStartingIndex(s, p), Solution().minStartingIndex(s, p[::-1])
    c = [x for x in (a, b) if x >= 0]
    return min(c) if c else -1

rng = random.Random(4)
for _ in range(300):
    n = rng.randint(2, 10)
    s = ''.join(rng.choice('ab') for _ in range(n)); p = ''.join(rng.choice('ab') for _ in range(rng.randint(1, n - 1)))
    exp = min([x for x in (brute(s, p), brute(s, p[::-1])) if x >= 0], default=-1)
    assert either_direction(s, p) == exp
    pre_ = match_arrays(s, p)[0]
    assert count_exact_kmp(s, p) == sum(1 for i in range(n) if pre_[i] >= len(p)) == \
        sum(1 for i in range(n - len(p) + 1) if s[i:i + len(p)] == p)
print('Bài 4 (KMP đếm khớp chính xác) và Bài 5 khớp vét cạn.')

Bài 4 (KMP đếm khớp chính xác) và Bài 5 khớp vét cạn.


Bài 4 cho thấy hai công cụ bổ sung cho nhau: prefix function trả về "đoạn dài nhất **kết thúc** tại `e` khớp với tiền tố pattern", còn Z-function trả về "khớp tiền tố **bắt đầu** tại `i`". Bài toán gốc cần đúng dạng thứ hai, nên Z-function là lựa chọn tự nhiên.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
z1 = Z(p + '#' + s);  z2 = Z(p[::-1] + '#' + s[::-1])
for i in range(n - m + 1):
    if z1[m + 1 + i] + z2[n - i + 1] >= m - 1: return i
return -1
```

**Checklist:**

- [ ] Có dấu phân cách `#` (khớp không vượt `m`)?
- [ ] Chỉ số hậu tố `n − i + 1` (kết thúc tại `i + m − 1`)?
- [ ] Điều kiện `≥ m − 1` (bao cả khớp hoàn toàn)?
- [ ] Duyệt `i` tăng dần để lấy chỉ số nhỏ nhất?
- [ ] So theo **vị trí**, không theo tập ký tự?